In [1]:
# Notebook imports
# Generated from standard/third-party imports used throughout this notebook.
import logging
import numpy as np
import os
import pyproj
import random
import rasterio
import shutil
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path
from tqdm import tqdm

In [2]:
LFM_ROOT = Path("/panfs/ccds02/nobackup/projects/lfm/")
DATA_ROOT = LFM_ROOT / "model_inputs/300_300_inputs/kaguya_static_all_wac"
ISEG = DATA_ROOT / "inst_seg"
ISEG_LABELS = ISEG / "labels"

label_files = list(ISEG_LABELS.glob("*.npz"))
example_label = label_files[0]
example_label_archive = np.load(example_label, allow_pickle=True)
label = example_label_archive
print(f"example label loaded: {example_label_archive}")
example_data = example_label_archive['mask']
example_bboxes = example_label_archive['bboxes']

print(example_data.shape, "\n", example_bboxes)

example label loaded: NpzFile '/panfs/ccds02/nobackup/projects/lfm/model_inputs/300_300_inputs/kaguya_static_all_wac/inst_seg/labels/M1108346760CE_r7950_c900_label.npz' with keys: mask, bboxes, num_craters
(300, 300) 
 [[283. 250.  12.  11.]
 [240. 266.  11.  11.]
 [ 40. 278.  19.  19.]
 [ 13. 261.  10.  10.]
 [192.  35.  12.  12.]
 [180.  26.  15.  16.]
 [110.  36.  14.  14.]
 [ 88. 147.  15.  15.]
 [ 62. 173.  28.  28.]
 [ 63. 206.  19.  18.]]


In [3]:
example_n_craters = example_label_archive['num_craters']
print(f"num_craters: {example_n_craters}, len(bboxes): {len(example_bboxes)}")

print(f"Unique mask values: {np.unique(label['mask'])}")

print('Datatypes of mask, bboxes, num_craters: ')
print(label["mask"].dtype)
print(label["bboxes"].dtype)
print(label["num_craters"].dtype)

num_craters: 10, len(bboxes): 10
Unique mask values: [ 0  1  2  3  4  5  6  7  8  9 10]
Datatypes of mask, bboxes, num_craters: 
uint8
float64
int64


## Create Raw IMP Semantic Segmentation Dataset

Converts the released IMP dataset from flat split directories into the standard semantic segmentation layout expected by the training scripts. Each `*_img.tif` file is copied into `{split}/chips` with the NAC chip suffix, and the matching `*_mask.tif` file is saved as a binary `.npy` label in `{split}/labels`. This section creates the raw IMP dataset variant with all source samples retained.

In [4]:
IMP_SOURCE_ROOT = Path("/explore/nobackup/projects/lfm/processed_data/Lunar/data_release/IMP_dataset")
IMP_OUTPUT_BASE_ROOT = Path("/explore/nobackup/projects/lfm/model_inputs/256_256_inputs")

IMP_VARIANTS = {
    "raw_all": {
        "output_root": IMP_OUTPUT_BASE_ROOT / "imp_sem_seg_raw",
        "remove_nodata": False,
        "target_only": False,
    },
    "clean_all": {
        "output_root": IMP_OUTPUT_BASE_ROOT / "imp_sem_seg_clean_all",
        "remove_nodata": True,
        "target_only": False,
    },
    "clean_targets_only": {
        "output_root": IMP_OUTPUT_BASE_ROOT / "imp_sem_seg_clean_targets_only",
        "remove_nodata": True,
        "target_only": True,
    },
    "raw_targets_only": {
        "output_root": IMP_OUTPUT_BASE_ROOT / "imp_sem_seg_raw_targets_only",
        "remove_nodata": False,
        "target_only": True,
    },
}

IMP_ACTIVE_VARIANT = "raw_all"
IMP_VARIANT = IMP_VARIANTS[IMP_ACTIVE_VARIANT]
IMP_OUTPUT_ROOT = IMP_VARIANT["output_root"]
IMP_REMOVE_NODATA = IMP_VARIANT["remove_nodata"]
IMP_TARGET_ONLY = IMP_VARIANT["target_only"]

print(f"IMP active variant: {IMP_ACTIVE_VARIANT}")
print(f"IMP output root: {IMP_OUTPUT_ROOT}")
print(f"IMP remove nodata samples: {IMP_REMOVE_NODATA}")
print(f"IMP target-only samples: {IMP_TARGET_ONLY}")

IMP_SPLITS = ["train", "val", "test"]
IMP_MASK_KIND = "mask"  # "mask" or "mask_orig"
IMP_MAX_WORKERS = 10
IMP_OVERWRITE = False
IMP_USE_MANIFESTS = False  # If True, restrict samples to {split}.txt stems.

IMP_IMAGE_SUFFIX = "_input_nac_chip"
IMP_LABEL_SUFFIX = "_label"


IMP active variant: raw_all
IMP output root: /explore/nobackup/projects/lfm/model_inputs/256_256_inputs/imp_sem_seg_raw
IMP remove nodata samples: False
IMP target-only samples: False


In [5]:
def read_manifest_stems(split: str) -> set[str] | None:
    if not IMP_USE_MANIFESTS:
        return None
    manifest_path = IMP_SOURCE_ROOT / f"{split}.txt"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Missing manifest for split {split}: {manifest_path}")
    return {
        line.strip()
        for line in manifest_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    }


def find_imp_samples(split: str) -> list[tuple[Path, Path, str]]:
    split_dir = IMP_SOURCE_ROOT / split
    allowed_stems = read_manifest_stems(split)
    samples = []
    missing_labels = []

    for image_path in sorted(split_dir.glob("*_img.tif")):
        sample_stem = image_path.name.removesuffix("_img.tif")
        if allowed_stems is not None and sample_stem not in allowed_stems:
            continue
        label_path = split_dir / f"{sample_stem}_{IMP_MASK_KIND}.tif"
        if not label_path.exists():
            missing_labels.append(label_path)
            continue
        samples.append((image_path, label_path, sample_stem))

    if missing_labels:
        raise FileNotFoundError(
            f"{split}: {len(missing_labels)} image(s) are missing labels; "
            f"first missing label: {missing_labels[0]}"
        )
    if not samples:
        raise ValueError(f"{split}: found no IMP samples under {split_dir}")
    return samples


def convert_imp_sample(args: tuple[str, Path, Path, str]) -> tuple[str, str, int]:
    split, image_path, label_path, sample_stem = args
    chips_dir = IMP_OUTPUT_ROOT / split / "chips"
    labels_dir = IMP_OUTPUT_ROOT / split / "labels"
    chips_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)

    chip_out = chips_dir / f"{sample_stem}{IMP_IMAGE_SUFFIX}.tif"
    label_out = labels_dir / f"{sample_stem}{IMP_LABEL_SUFFIX}.npy"

    if IMP_OVERWRITE or not chip_out.exists():
        shutil.copy2(image_path, chip_out)

    rewrite_label = IMP_OVERWRITE or not label_out.exists() or label_out.stat().st_size == 0
    if not rewrite_label:
        try:
            mask = np.load(label_out)
        except (EOFError, OSError, ValueError):
            rewrite_label = True

    if rewrite_label:
        with rasterio.open(label_path) as src:
            mask = src.read(1)
        mask = np.where(mask > 0, 1, 0).astype(np.uint8)
        np.save(label_out, mask)

    return split, sample_stem, int(np.count_nonzero(mask))


def create_imp_sem_seg_dataset() -> None:
    all_jobs = []
    for split in IMP_SPLITS:
        samples = find_imp_samples(split)
        print(f"{split}: found {len(samples)} image/mask sample(s)")
        all_jobs.extend((split, image_path, label_path, sample_stem) for image_path, label_path, sample_stem in samples)

    counts = {split: {"samples": 0, "positive_samples": 0} for split in IMP_SPLITS}
    with ProcessPoolExecutor(max_workers=IMP_MAX_WORKERS) as executor:
        results = executor.map(convert_imp_sample, all_jobs, chunksize=32)
        for split, _, positive_pixels in tqdm(results, total=len(all_jobs), desc="Converting IMP samples"):
            counts[split]["samples"] += 1
            counts[split]["positive_samples"] += int(positive_pixels > 0)

    print(f"Wrote IMP semantic segmentation dataset to: {IMP_OUTPUT_ROOT}")
    for split in IMP_SPLITS:
        chips = sorted((IMP_OUTPUT_ROOT / split / "chips").glob("*.tif"))
        labels = sorted((IMP_OUTPUT_ROOT / split / "labels").glob("*.npy"))
        print(
            f"{split}: {len(chips)} chips, {len(labels)} .npy labels, "
            f"{counts[split]['positive_samples']} positive sample(s)"
        )


In [ ]:
# Run when ready.
create_imp_sem_seg_dataset()

In [ ]:
# Quick output sanity check after conversion.
for split in IMP_SPLITS:
    chips = sorted((IMP_OUTPUT_ROOT / split / "chips").glob("*.tif"))
    labels = sorted((IMP_OUTPUT_ROOT / split / "labels").glob("*.npy"))
    print(f"\n{split}: {len(chips)} chips, {len(labels)} labels")
    if chips and labels:
        with rasterio.open(chips[0]) as src:
            image = src.read([1])[0]
        label = np.load(labels[0])
        print("  sample chip:", chips[0].name, image.shape, image.dtype)
        print("  sample label:", labels[0].name, label.shape, label.dtype, np.unique(label).tolist())

In [ ]:
a

## Create 7 band dataset
Creates `data_7band/{split}/{chips,labels}` from `data/{split}/{chips,labels}`. Chip TIFFs keep only bands 1-7. Labels are copied unchanged.

In [ ]:

try:
    import pyproj
    os.environ["PROJ_LIB"] = pyproj.datadir.get_data_dir()
except Exception:
    pass

logging.getLogger("rasterio._env").setLevel(logging.ERROR)


SRC_ROOT = Path("data")
DST_ROOT = Path("data_7band")
MAX_WORKERS = 16


def write_7band_chip(args):
    chip_path, out_path = args
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(chip_path) as src:
        if src.count < 7:
            raise ValueError(f"{chip_path} has only {src.count} bands")

        profile = src.profile.copy()
        profile.update(driver="GTiff", count=7)
        data = src.read(indexes=list(range(1, 8)))

        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(data)

    return str(out_path)


def copy_label(args):
    label_path, out_path = args
    out_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(label_path, out_path)
    return str(out_path)


def process_split(split):
    src_chips = SRC_ROOT / split / "chips"
    src_labels = SRC_ROOT / split / "labels"
    dst_chips = DST_ROOT / split / "chips"
    dst_labels = DST_ROOT / split / "labels"

    chip_jobs = [(p, dst_chips / p.name) for p in sorted(src_chips.glob("*.tif"))]
    label_jobs = [(p, dst_labels / p.name) for p in sorted(src_labels.iterdir()) if p.is_file()]

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        chip_outputs = list(executor.map(write_7band_chip, chip_jobs))

    with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
        label_outputs = list(executor.map(copy_label, label_jobs))

    print(f"{split}: wrote {len(chip_outputs)} chips and copied {len(label_outputs)} labels")

In [ ]:
# for split in ["train"]:
#     process_split(split)

In [ ]:
# for split in ["val"]:
#     process_split(split)

In [ ]:
# for split in ["test"]:
#     process_split(split)

## Create Semantic Segmentation Splits

Builds a fresh `train`/`val`/`test` split from `kaguya_static_all_wac/sem_seg`. Chip TIFFs are saved with only the first 7 bands. Labels are copied unchanged; the label-processing helper is kept as a no-op hook.

In [ ]:

LFM_ROOT = Path("/explore/nobackup/projects/lfm/model_inputs/300_300_inputs")
SEM_SEG_ROOT = LFM_ROOT / "kaguya_static_all_wac" / "sem_seg"
SEM_SEG_OUTPUT_ROOT = LFM_ROOT / "full_model_sem_seg_v2"

SEM_SEG_CHIPS_DIR = SEM_SEG_ROOT / "chips"
SEM_SEG_LABELS_DIR = SEM_SEG_ROOT / "labels"

SEM_SEG_IMAGE_GLOB = "*.tif"
SEM_SEG_LABEL_GLOB = "*_label.*"
SEM_SEG_IMAGE_SUFFIX = "_input_wac_static_chip"
SEM_SEG_LABEL_SUFFIX = "_label"

SEM_SEG_SEED = 42
SEM_SEG_N_TEST = 100
SEM_SEG_TRAIN_FRACTION = 0.95
SEM_SEG_MAX_WORKERS = 16


def split_key(path, suffix):
    stem = path.stem
    if suffix and stem.endswith(suffix):
        return stem[: -len(suffix)]
    return stem


def find_sem_seg_pairs():
    chips = {
        split_key(path, SEM_SEG_IMAGE_SUFFIX): path
        for path in sorted(SEM_SEG_CHIPS_DIR.glob(SEM_SEG_IMAGE_GLOB))
    }
    labels = {
        split_key(path, SEM_SEG_LABEL_SUFFIX): path
        for path in sorted(SEM_SEG_LABELS_DIR.glob(SEM_SEG_LABEL_GLOB))
    }

    keys = sorted(set(chips) & set(labels))
    chip_only = sorted(set(chips) - set(labels))
    label_only = sorted(set(labels) - set(chips))

    print(f"matched pairs: {len(keys)}")
    print(f"chips only:    {len(chip_only)}")
    print(f"labels only:   {len(label_only)}")

    if len(keys) <= SEM_SEG_N_TEST:
        raise ValueError(f"Need more than {SEM_SEG_N_TEST} pairs, found {len(keys)}")

    return [(chips[key], labels[key]) for key in keys]


def process_sem_seg_label(label_path: Path, label_out: Path) -> None:
    label_out.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(label_path, label_out)


def write_sem_seg_7band_chip(chip_path: Path, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(chip_path) as src:
        if src.count < 7:
            raise ValueError(f"{chip_path} has only {src.count} bands")
        profile = src.profile.copy()
        profile.update(driver="GTiff", count=7)
        data = src.read(indexes=list(range(1, 8)))
        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(data)


def copy_sem_seg_pair_job(args):
    chip_path, label_path, split = args
    chip_out = SEM_SEG_OUTPUT_ROOT / split / "chips" / chip_path.name
    label_out = SEM_SEG_OUTPUT_ROOT / split / "labels" / label_path.name
    write_sem_seg_7band_chip(chip_path, chip_out)
    process_sem_seg_label(label_path, label_out)
    return split


def create_sem_seg_split():
    pairs = find_sem_seg_pairs()
    rng = random.Random(SEM_SEG_SEED)
    rng.shuffle(pairs)

    test_pairs = pairs[:SEM_SEG_N_TEST]
    remaining = pairs[SEM_SEG_N_TEST:]
    n_train = int(round(len(remaining) * SEM_SEG_TRAIN_FRACTION))

    split_pairs = {
        "train": remaining[:n_train],
        "val": remaining[n_train:],
        "test": test_pairs,
    }

    jobs = [
        (chip_path, label_path, split)
        for split, pairs_for_split in split_pairs.items()
        for chip_path, label_path in pairs_for_split
    ]

    with ProcessPoolExecutor(max_workers=SEM_SEG_MAX_WORKERS) as executor:
        list(executor.map(copy_sem_seg_pair_job, jobs))

    for split, pairs_for_split in split_pairs.items():
        print(f"{split}: {len(pairs_for_split)} pairs")
    print(f"wrote split dataset to: {SEM_SEG_OUTPUT_ROOT.resolve()}")

In [ ]:

LFM_ROOT = Path("/explore/nobackup/projects/lfm/model_inputs/300_300_inputs")
SEM_SEG_ROOT = LFM_ROOT / "kaguya_static_all_wac" / "sem_seg"
SEM_SEG_OUTPUT_ROOT = LFM_ROOT / "full_model_sem_seg_v2"

SEM_SEG_CHIPS_DIR = SEM_SEG_ROOT / "chips"
SEM_SEG_LABELS_DIR = SEM_SEG_ROOT / "labels"

SEM_SEG_IMAGE_GLOB = "*.tif"
SEM_SEG_LABEL_GLOB = "*_label.*"
SEM_SEG_IMAGE_SUFFIX = "_input_wac_static_chip"
SEM_SEG_LABEL_SUFFIX = "_label"

SEM_SEG_SEED = 42
SEM_SEG_N_TEST = 100
SEM_SEG_TRAIN_FRACTION = 0.95
SEM_SEG_MAX_WORKERS = 16


def split_key(path, suffix):
    stem = path.stem
    if suffix and stem.endswith(suffix):
        return stem[: -len(suffix)]
    return stem


def find_sem_seg_pairs():
    chips = {
        split_key(path, SEM_SEG_IMAGE_SUFFIX): path
        for path in sorted(SEM_SEG_CHIPS_DIR.glob(SEM_SEG_IMAGE_GLOB))
    }
    labels = {
        split_key(path, SEM_SEG_LABEL_SUFFIX): path
        for path in sorted(SEM_SEG_LABELS_DIR.glob(SEM_SEG_LABEL_GLOB))
    }

    keys = sorted(set(chips) & set(labels))
    chip_only = sorted(set(chips) - set(labels))
    label_only = sorted(set(labels) - set(chips))

    print(f"matched pairs: {len(keys)}")
    print(f"chips only:    {len(chip_only)}")
    print(f"labels only:   {len(label_only)}")

    if len(keys) <= SEM_SEG_N_TEST:
        raise ValueError(f"Need more than {SEM_SEG_N_TEST} pairs, found {len(keys)}")

    return [(chips[key], labels[key]) for key in keys]


def process_sem_seg_label(label_path: Path, label_out: Path) -> None:
    label_out.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(label_path, label_out)


def write_sem_seg_7band_chip(chip_path: Path, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(chip_path) as src:
        if src.count < 7:
            raise ValueError(f"{chip_path} has only {src.count} bands")
        profile = src.profile.copy()
        profile.update(driver="GTiff", count=7)
        data = src.read(indexes=list(range(1, 8)))
        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(data)


def copy_sem_seg_pair_job(args):
    chip_path, label_path, split = args
    chip_out = SEM_SEG_OUTPUT_ROOT / split / "chips" / chip_path.name
    label_out = SEM_SEG_OUTPUT_ROOT / split / "labels" / label_path.name
    write_sem_seg_7band_chip(chip_path, chip_out)
    process_sem_seg_label(label_path, label_out)
    return split


def create_sem_seg_split():
    pairs = find_sem_seg_pairs()
    rng = random.Random(SEM_SEG_SEED)
    rng.shuffle(pairs)

    test_pairs = pairs[:SEM_SEG_N_TEST]
    remaining = pairs[SEM_SEG_N_TEST:]
    n_train = int(round(len(remaining) * SEM_SEG_TRAIN_FRACTION))

    split_pairs = {
        "train": remaining[:n_train],
        "val": remaining[n_train:],
        "test": test_pairs,
    }

    jobs = [
        (chip_path, label_path, split)
        for split, pairs_for_split in split_pairs.items()
        for chip_path, label_path in pairs_for_split
    ]

    with ProcessPoolExecutor(max_workers=SEM_SEG_MAX_WORKERS) as executor:
        list(executor.map(copy_sem_seg_pair_job, jobs))

    for split, pairs_for_split in split_pairs.items():
        print(f"{split}: {len(pairs_for_split)} pairs")
    print(f"wrote split dataset to: {SEM_SEG_OUTPUT_ROOT.resolve()}")

In [ ]:
create_sem_seg_split()

## Create Instance Segmentation Splits

Builds a fresh `train`/`val`/`test` split from the instance-segmentation source directory. Chip TIFFs are saved with only the first 7 bands. `.npz` labels are copied unchanged.

In [ ]:

ISEG_SOURCE_ROOT = Path("/explore/nobackup/projects/lfm/model_inputs/300_300_inputs/kaguya_static_all_wac/inst_seg")
ISEG_OUTPUT_ROOT = Path("/explore/nobackup/projects/lfm/model_inputs/300_300_inputs/full_model_inst_seg_v2")

ISEG_CHIPS_DIR = ISEG_SOURCE_ROOT / "chips"
ISEG_LABELS_DIR = ISEG_SOURCE_ROOT / "labels"

ISEG_IMAGE_GLOB = "*.tif"
ISEG_LABEL_GLOB = "*_label.npz"
ISEG_IMAGE_SUFFIX = "_input_wac_static_chip"
ISEG_LABEL_SUFFIX = "_label"

ISEG_SEED = 42
ISEG_N_TEST = 100
ISEG_TRAIN_FRACTION = 0.95
ISEG_MAX_WORKERS = 16


def split_key(path, suffix):
    stem = path.stem
    if suffix and stem.endswith(suffix):
        return stem[: -len(suffix)]
    return stem


def find_iseg_pairs():
    chips = {
        split_key(path, ISEG_IMAGE_SUFFIX): path
        for path in sorted(ISEG_CHIPS_DIR.glob(ISEG_IMAGE_GLOB))
    }
    labels = {
        split_key(path, ISEG_LABEL_SUFFIX): path
        for path in sorted(ISEG_LABELS_DIR.glob(ISEG_LABEL_GLOB))
    }

    keys = sorted(set(chips) & set(labels))
    chip_only = sorted(set(chips) - set(labels))
    label_only = sorted(set(labels) - set(chips))

    print(f"matched pairs: {len(keys)}")
    print(f"chips only:    {len(chip_only)}")
    print(f"labels only:   {len(label_only)}")

    if len(keys) <= ISEG_N_TEST:
        raise ValueError(f"Need more than {ISEG_N_TEST} pairs, found {len(keys)}")

    return [(chips[key], labels[key]) for key in keys]


def write_iseg_7band_chip(chip_path: Path, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with rasterio.open(chip_path) as src:
        if src.count < 7:
            raise ValueError(f"{chip_path} has only {src.count} bands")
        profile = src.profile.copy()
        profile.update(driver="GTiff", count=7)
        data = src.read(indexes=list(range(1, 8)))
        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(data)


def copy_iseg_pair_job(args):
    chip_path, label_path, split = args
    chip_out = ISEG_OUTPUT_ROOT / split / "chips" / chip_path.name
    label_out = ISEG_OUTPUT_ROOT / split / "labels" / label_path.name
    write_iseg_7band_chip(chip_path, chip_out)
    label_out.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(label_path, label_out)
    return split


def create_iseg_split():
    pairs = find_iseg_pairs()
    rng = random.Random(ISEG_SEED)
    rng.shuffle(pairs)

    test_pairs = pairs[:ISEG_N_TEST]
    remaining = pairs[ISEG_N_TEST:]
    n_train = int(round(len(remaining) * ISEG_TRAIN_FRACTION))

    split_pairs = {
        "train": remaining[:n_train],
        "val": remaining[n_train:],
        "test": test_pairs,
    }

    jobs = [
        (chip_path, label_path, split)
        for split, pairs_for_split in split_pairs.items()
        for chip_path, label_path in pairs_for_split
    ]

    with ProcessPoolExecutor(max_workers=ISEG_MAX_WORKERS) as executor:
        list(executor.map(copy_iseg_pair_job, jobs))

    for split, pairs_for_split in split_pairs.items():
        print(f"{split}: {len(pairs_for_split)} pairs")
    print(f"wrote split dataset to: {ISEG_OUTPUT_ROOT.resolve()}")

In [ ]:
create_iseg_split()

In [ ]:
for split in ["train", "val", "test"]:
    chips = sorted((ISEG_OUTPUT_ROOT / split / "chips").glob("*.tif"))
    labels = sorted((ISEG_OUTPUT_ROOT / split / "labels").glob("*.npz"))
    other_labels = [p for p in (ISEG_OUTPUT_ROOT / split / "labels").iterdir() if p.is_file() and p.suffix != ".npz"]
    print(f"{split}: {len(chips)} chips, {len(labels)} .npz labels, {len(other_labels)} non-npz labels")